# GPNAM: Gaussian-process-inspired additive model

GPNAM fixes a random Fourier feature map for each scalar input and estimates only additive coefficients. It approximates RBF-kernel functions but does not produce GP posterior covariance.


## Model


$$
\phi_j(x_j)=\sqrt{\frac{2}{M}}
\left[\cos\!\left(z_mx_j/\ell_j+c_{mj}\right)\right]_{m=1}^{M},
\qquad
\eta(x)=\beta_0+\sum_j\phi_j(x_j)^\top w_j.
$$

Selected pairs add two-dimensional GP-NA2M feature maps.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_preprocessing` and `categorical_preprocessing` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import GPNAMClassifier, GPNAMLSS, GPNAMRegressor


model = GPNAMRegressor(
    rff_num_feat=32,
    kernel_width="auto",
    solver="cg",
    ridge=0.05,
    interactions=(("x1", "x2"),),
    rff_random_state=7,
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

Regression defaults to the fixed conjugate-gradient ridge solve. `basis_transform`, `basis_metadata`, and `model_complexity` expose the fitted finite basis.


In [ ]:
if RUN_TRAINING:
    Phi = model.basis_transform(X_test)
    display(Phi.shape)
    display(model.basis_metadata())
    display(model.model_complexity())
    display(model.kernel_widths_)


## Task variants and limits

`GPNAMClassifier` and `GPNAMLSS` use gradient training. Distributional uncertainty is aleatoric, not GP posterior uncertainty.
